# 🎙️ WAXAL ASR — Gemma 3n on a free Colab T4

Fine-tunes **google/gemma-3n-E2B-it** with LoRA for one language at a time
(Lingala / Shona / Luganda), built for a **free T4 (16 GB, ~4h sessions)**.

**The T4 survival strategy (read this):**
- Train **one language at a time**. Set `LANGUAGE` below, run top to bottom.
- Checkpoints save to **Google Drive** every N steps, so a disconnect never
  loses progress — just re-run and it resumes.
- We cap training examples (`MAX_TRAIN`) so a run finishes inside one session.
- After all three languages are trained, run the **Inference** section to build
  `submission.csv` in the exact Zindi format.

**Runtime → Change runtime type → T4 GPU** before you start.


## 1. Setup — mount Drive & install deps

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# All outputs live here so nothing is lost on disconnect
WORK_DIR = '/content/drive/MyDrive/waxal'
import os
os.makedirs(WORK_DIR, exist_ok=True)
print('Working dir:', WORK_DIR)

Mounted at /content/drive
Working dir: /content/drive/MyDrive/waxal


In [2]:
%%capture
!pip install -q -U \
    "transformers>=4.52.0" datasets peft trl jiwer \
    librosa soundfile accelerate bitsandbytes timm huggingface_hub
print("deps installed")

## 2. Hugging Face login

Gemma is gated: open https://huggingface.co/google/gemma-3n-E2B-it once and
click **Agree**. Then make a **read** token at
https://huggingface.co/settings/tokens and paste it below.

In [3]:
from huggingface_hub import login
login()  # paste your read token when prompted

## 3. Config — set LANGUAGE, then run everything below

In [4]:
from typing import Final

MODEL_ID: Final = "google/gemma-3n-E2B-it"
DATASET_ID: Final = "google/WaxalNLP"

# >>> CHANGE THIS for each run: "lin", "sna", or "lug" <<<
LANGUAGE: Final = "lin"

# T4 budget knobs -------------------------------------------------
MAX_TRAIN: Final   = 300   # cap train examples so a run finishes in one session
MAX_EVAL: Final    = 200    # local validation size
MAX_STEPS: Final   = 600    # hard stop; raise if your session has time left
SAVE_EVERY: Final  = 100    # checkpoint cadence (to Drive)
BATCH: Final       = 1      # T4 is tight; keep at 1
GRAD_ACCUM: Final  = 8      # effective batch = BATCH * GRAD_ACCUM
MAX_NEW_TOKENS: Final = 128 # inference decode length
# -----------------------------------------------------------------

OUT_DIR = f"{WORK_DIR}/gemma-{LANGUAGE}"
import os; os.makedirs(OUT_DIR, exist_ok=True)
print("Training language:", LANGUAGE, "| output ->", OUT_DIR)

Training language: lin | output -> /content/drive/MyDrive/waxal/gemma-lin


## 4. Load the WAXAL audio for this language

The audio is streamed from Hugging Face — no full download. Each example has
`id`, `audio` (16 kHz), `transcription`, `language`.

The exact **config name** for the ASR data can vary (e.g. `lin` vs `lin_asr`).
This cell probes a few candidates and uses the first that loads, printing which
one worked so you know what happened.

In [5]:
from datasets import load_dataset, get_dataset_config_names, Audio, Dataset

# Find the right config for this language
try:
    configs = get_dataset_config_names(DATASET_ID)
    print("configs (first 40):", configs[:40])
except Exception as e:
    configs = []; print("could not list configs:", e)

candidates = [c for c in configs if c == LANGUAGE or c.startswith(LANGUAGE)]
candidates = sorted(candidates, key=lambda c: ("tts" in c, "asr" not in c, len(c)))
if not candidates:
    candidates = [LANGUAGE, f"{LANGUAGE}_asr"]
print("trying:", candidates)

CFG = None
for cfg in candidates:
    try:
        _ = load_dataset(DATASET_ID, cfg, split="train", streaming=True)
        CFG = cfg; print(f"✅ using config '{cfg}'"); break
    except Exception as e:
        print(f"  '{cfg}' failed: {str(e)[:100]}")
assert CFG is not None, "no config worked — check the dataset card"

def take_stream(split, n):
    """Stream `split`, keep first n examples, resample to 16 kHz."""
    s = load_dataset(DATASET_ID, CFG, split=split, streaming=True)
    s = s.cast_column("audio", Audio(sampling_rate=16_000))
    rows = []
    for i, ex in enumerate(s):
        if i >= n: break
        rows.append({"id": ex["id"],
                     "audio": ex["audio"]["array"],
                     "transcription": ex["transcription"]})
        if (i + 1) % 100 == 0:
            print(f"  {split}: {i+1}/{n} clips streamed")
    return Dataset.from_list(rows)

train_ds = take_stream("train", MAX_TRAIN)
try:
    eval_ds = take_stream("validation", MAX_EVAL)
except Exception:
    eval_ds = take_stream("train", MAX_EVAL)

print("train:", len(train_ds), "| eval:", len(eval_ds))
print("sample:", train_ds[0]["transcription"][:120])


README.md:   0%|          | 0.00/29.2k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

configs (first 40): ['ach_asr', 'ach_tts', 'aka_asr', 'amh_asr', 'bau_tts', 'dag_asr', 'dga_asr', 'ewe_asr', 'ewe_tts', 'fat_tts', 'ful_asr', 'ful_tts', 'hau_tts', 'ibo_tts', 'kik_tts', 'kpo_asr', 'lin_asr', 'lug_asr', 'lug_tts', 'luo_tts', 'mas_asr', 'mlg_asr', 'nyn_asr', 'nyn_tts', 'orm_asr', 'pcm_tts', 'sid_asr', 'sna_asr', 'tir_asr', 'sog_asr', 'swa_tts', 'twi_tts', 'yor_tts', 'wal_asr']
trying: ['lin_asr']


Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

...

Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

  train: 100/300 clips streamed
  train: 200/300 clips streamed
  train: 300/300 clips streamed


Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/40 [00:00<?, ?it/s]

  validation: 100/200 clips streamed
  validation: 200/200 clips streamed
train: 300 | eval: 200
sample: Ndaku oyo ezali na bosoto mpo bazali kosala ngo likolo ezali na mwinda misatu ekangami esika moko na se ezali na mabaya 


## 5. Text normalization — matters because CER is 50% of your score

In [6]:
import re

# Inspect a few real transcripts, then decide what to strip. WAXAL keeps
# casing/punctuation; the safest baseline is to match the references as-is and
# only normalize whitespace. If you strip punctuation here you MUST strip it
# the same way in references when scoring, or numbers will look worse/better
# than reality.
def normalize_text(t: str) -> str:
    t = t.strip()
    t = re.sub(r"\s+", " ", t)
    return t

for i in range(3):
    print(repr(train_ds[i]["transcription"]))

'Ndaku oyo ezali na bosoto mpo bazali kosala ngo likolo ezali na mwinda misatu ekangami esika moko na se ezali na mabaya oyo ba matelaka.'
'Mobali afandi alati kanzaku,ya langi ya motane na singa ya kingo ya mayaka alati ekoti ya ba silamu.'
'Opena nde nazomona ezali on dirait ezali pondu ya kotuta pondu yango kasi eza ya kotekisama kasi yango bozomona bien pondu yango on dirait ezali na langi moko ya mai ya pondu ndenge na ndenge kaka bozomona'


## 6. Chat formatting for Gemma 3n (audio in, text out)

In [7]:
SYSTEM_MESSAGE = "You are an expert transcriber of African-language speech."
USER_MESSAGE = "Transcribe the audio exactly, word for word."

def collate_fn(examples):
    texts, audios = [], []
    for ex in examples:
        msgs = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_MESSAGE}]},
            {"role":"user","content":[
                {"type":"audio","audio":ex["audio"]},
                {"type":"text","text":USER_MESSAGE}]},
            {"role":"assistant","content":[
                {"type":"text","text":normalize_text(ex["transcription"])}]},
        ]
        text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False).strip()
        texts.append(text)
        audios.append(ex["audio"])   # raw 16 kHz array

    batch = processor(text=texts, audio=audios, return_tensors="pt", padding=True)

    labels = batch["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    for attr in ("audio_token_id", "image_token_id", "boi_token_id", "eoi_token_id"):
        tok = getattr(processor.tokenizer, attr, None)
        if tok is not None:
            labels[labels == tok] = -100
    batch["labels"] = labels
    return batch

# The trainer will get train_ds / eval_ds directly (raw), so nothing to precompute here.
print("collator ready; train rows:", len(train_ds))

collator ready; train rows: 300


## 7. Load Gemma 3n in 4-bit + LoRA

In [12]:
import timm  # must import before transformers loads Gemma3n
import torch, transformers
from transformers import BitsAndBytesConfig

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True, # Enable offloading of 32-bit modules to CPU/disk
)

processor = transformers.AutoProcessor.from_pretrained(MODEL_ID)
processor.tokenizer.padding_side = "right"

model = transformers.Gemma3nForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb, torch_dtype=torch.bfloat16, device_map="auto",
)
if hasattr(model.config, "altup_coef_clip"):
    model.config.altup_coef_clip = None
# some versions store it on the text config
if hasattr(model.config, "text_config") and hasattr(model.config.text_config, "altup_coef_clip"):
    model.config.text_config.altup_coef_clip = None
model.config.use_cache = False
print("loaded on", model.device)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1526 [00:00<?, ?it/s]

StrictDataclassFieldValidationError: Validation error for field 'altup_coef_clip':
    TypeError: Field 'altup_coef_clip' expected float, got NoneType (value: None)

In [9]:
import peft
peft_config = peft.LoraConfig(
    task_type="CAUSAL_LM", r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

## 8. Train — checkpoints to Drive; resumes automatically

In [10]:
import trl, glob

args = trl.SFTConfig(
    output_dir=OUT_DIR,
    per_device_train_batch_size=BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    max_steps=MAX_STEPS,
    save_steps=SAVE_EVERY,
    logging_steps=10,
    learning_rate=2e-4,
    warmup_steps=10,
    lr_scheduler_type="cosine",
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_8bit",
    save_total_limit=2,
    report_to="none",
    seed=42,
    remove_unused_columns=False,   # keep 'audio' + 'transcription' for the collator
    dataset_kwargs={"skip_prepare_dataset": True},
    dataset_text_field="",
)

trainer = trl.SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,        # raw dataset; collator tokenizes each batch
    data_collator=collate_fn,
    peft_config=peft_config,
    processing_class=processor.tokenizer,
)

ckpts = sorted(glob.glob(f"{OUT_DIR}/checkpoint-*"))
resume = ckpts[-1] if ckpts else None
print("resuming from:", resume)
trainer.train(resume_from_checkpoint=resume)

trainer.save_model(f"{OUT_DIR}/final")
processor.save_pretrained(f"{OUT_DIR}/final")
print("saved ->", f"{OUT_DIR}/final")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


resuming from: None


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


RuntimeError: result type Float can't be cast to the desired output type unsigned char

## 9. Quick local WER/CER check

Sanity-check on the held-out eval set before spending submissions. This mirrors
the leaderboard: score = 0.5·WER + 0.5·CER (lower is better).

In [ ]:
import jiwer, numpy as np

def transcribe_one(example):
    msgs = [
        {"role":"system","content":[{"type":"text","text":SYSTEM_MESSAGE}]},
        {"role":"user","content":[
            {"type":"audio","audio":example["audio"]["array"]},
            {"type":"text","text":USER_MESSAGE},
        ]},
    ]
    prompt = processor.tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    inputs = processor(text=prompt, audio=[np.asarray(example["audio"]["array"]).flatten()],
                       return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                             pad_token_id=processor.tokenizer.pad_token_id)
    text = processor.tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return normalize_text(text)

preds, refs = [], []
for ex in eval_ds.select(range(min(40, len(eval_ds)))):
    preds.append(transcribe_one(ex)); refs.append(normalize_text(ex["transcription"]))

wer = jiwer.wer(refs, preds); cer = jiwer.cer(refs, preds)
print(f"WER={wer:.3f}  CER={cer:.3f}  score={0.5*wer+0.5*cer:.3f}")
print("PRED:", preds[0][:100]); print("REF :", refs[0][:100])

---
# 🏁 INFERENCE → submission.csv

Run this section **only after** you've trained all three languages
(`gemma-lin/final`, `gemma-sna/final`, `gemma-lug/final` all exist in Drive).

It reads your uploaded `Test.csv` + `SampleSubmission.csv`, routes each test ID
to the matching language adapter, transcribes, and writes `submission.csv` in
the exact `ID,Target` format Zindi expects.

In [ ]:
# Upload Test.csv and SampleSubmission.csv to the Colab session first
# (left sidebar → Files → upload), or point these paths at Drive copies.
import pandas as pd
TEST_CSV = "Test.csv"
SAMPLE_CSV = "SampleSubmission.csv"

test = pd.read_csv(TEST_CSV)
id_col = test.columns[0]  # 'ID'
test["lang"] = test[id_col].str.split("_").str[0]
print(test["lang"].value_counts())

In [ ]:
# We need the raw test audio. The dataset's unlabeled/test split contains it;
# build an id -> audio lookup per language from the HF dataset.
from datasets import load_dataset, get_dataset_config_names, Audio

def load_lang_audio_lookup(lang):
    configs = get_dataset_config_names(DATASET_ID)
    cands = sorted([c for c in configs if c.startswith(lang)],
                   key=lambda c: ("tts" in c, "asr" not in c, len(c))) or [lang]
    d = None
    for cfg in cands:
        try:
            d = load_dataset(DATASET_ID, cfg); break
        except Exception: pass
    # merge every split so we can find any test id
    lookup = {}
    for split in d:
        dd = d[split].cast_column("audio", Audio(sampling_rate=16_000))
        for ex in dd:
            lookup[ex["id"]] = ex["audio"]["array"]
    return lookup

In [ ]:
import gc, numpy as np, torch, transformers, peft

def load_adapter(lang):
    base = transformers.Gemma3nForConditionalGeneration.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
    m = peft.PeftModel.from_pretrained(base, f"{WORK_DIR}/gemma-{lang}/final")
    m.eval()
    proc = transformers.AutoProcessor.from_pretrained(f"{WORK_DIR}/gemma-{lang}/final")
    return m, proc

def infer(lang, ids, audio_lookup, m, proc):
    rows = []
    for _id in ids:
        arr = audio_lookup.get(_id)
        if arr is None:
            rows.append((_id, "")); continue
        msgs = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_MESSAGE}]},
            {"role":"user","content":[
                {"type":"audio","audio":arr},
                {"type":"text","text":USER_MESSAGE}]},
        ]
        prompt = proc.tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
        inp = proc(text=prompt, audio=[np.asarray(arr).flatten()],
                   return_tensors="pt", padding=True).to(m.device)
        with torch.no_grad():
            out = m.generate(**inp, max_new_tokens=MAX_NEW_TOKENS,
                             pad_token_id=proc.tokenizer.pad_token_id)
        txt = proc.tokenizer.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
        rows.append((_id, txt.strip()))
    return rows

In [ ]:
all_rows = []
for lang in ["lin", "sna", "lug"]:
    ids = test.loc[test["lang"] == lang, id_col].tolist()
    if not ids:
        continue
    print(f"{lang}: {len(ids)} clips — loading audio lookup…")
    lookup = load_lang_audio_lookup(lang)
    m, proc = load_adapter(lang)
    all_rows += infer(lang, ids, lookup, m, proc)
    del m, proc, lookup; gc.collect(); torch.cuda.empty_cache()
    print(f"{lang} done ({len(all_rows)} total so far)")

In [ ]:
# Write submission in the SampleSubmission format
sample = pd.read_csv(SAMPLE_CSV)
id_c, tgt_c = sample.columns[0], sample.columns[1]  # 'ID', 'Target'
pred_map = dict(all_rows)
sample[tgt_c] = sample[id_c].map(pred_map).fillna("")
out_path = f"{WORK_DIR}/submission.csv"
sample.to_csv(out_path, index=False)
sample.to_csv("submission.csv", index=False)  # also to session for easy download
print("wrote", out_path, "rows:", len(sample))
sample.head()